In [1]:
# ITERATION 3: Slim feature set based on permutation importance
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

# --- Load data ---
df = pd.read_csv('../raw_data/features_enriched_data.csv')
print(f"Loaded: {df.shape}")

# --- Create binary target (same as iteration 2) ---
df = df[df['status_enriched'] != 'unknown'].copy()
df['target'] = df['status_enriched'].map({
    'operating': 1, 'acquired': 1, 'closed': 0
})
print(f"After dropping unknown: {df.shape}")
print(f"Class balance:\n{df['target'].value_counts(normalize=True).round(3)}")

# --- Slim feature set ---
slim_features = [
    'age_first_funding_days',
    'funding_span_days',
    'avg_raised_per_round',
    'has_multiple_rounds',
    'industry_group',
    'region_group',
]

num_cols = ['age_first_funding_days', 'funding_span_days', 'avg_raised_per_round']
bin_cols = ['has_multiple_rounds']
cat_cols = ['industry_group', 'region_group']

X = df[slim_features].copy()
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

Loaded: (44363, 31)
After dropping unknown: (43212, 32)
Class balance:
target
1    0.745
0    0.255
Name: proportion, dtype: float64

Train: (34569, 6), Test: (8643, 6)


In [2]:
# --- Preprocessor ---
preprocessor = ColumnTransformer([
    ('num', RobustScaler(), num_cols),
    ('bin', 'passthrough', bin_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])

In [5]:
# --- Models ---
models = {
    'LogReg (balanced)': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Decision Tree (balanced)': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest (balanced)': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
}

In [6]:
# --- Train and evaluate ---
print("=" * 75)
print("ITERATION 3: SLIM FEATURE SET (6 features)")
print("=" * 75)
print(f"{'Model':<28} {'CV AUC':>10} {'Test AUC':>10} {'F1 closed':>10} {'Accuracy':>10}")
print("-" * 75)

results = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    test_auc = roc_auc_score(y_test, y_proba)
    f1_closed = f1_score(y_test, y_pred, pos_label=0)
    acc = accuracy_score(y_test, y_pred)

    results[name] = {'cv_auc': cv_scores.mean(), 'test_auc': test_auc,
                     'f1_closed': f1_closed, 'accuracy': acc, 'pipe': pipe}
    print(f"{name:<28} {cv_scores.mean():>10.4f} {test_auc:>10.4f} {f1_closed:>10.4f} {acc:>10.4f}")


ITERATION 3: SLIM FEATURE SET (6 features)
Model                            CV AUC   Test AUC  F1 closed   Accuracy
---------------------------------------------------------------------------
LogReg (balanced)                0.6095     0.6086     0.4124     0.5483
Decision Tree (balanced)         0.5221     0.5187     0.3005     0.6207
Random Forest (balanced)         0.5554     0.5600     0.2563     0.6904
KNN (k=5)                        0.5492     0.5530     0.2153     0.6964
Gradient Boosting                0.6173     0.6211     0.0107     0.7444
XGBoost                          0.5924     0.5986     0.0813     0.7411
